# Notebook 3 of 3: deriving every well-level variable (163 original Northeast wells)

| Cell | Variable(s) | Source |
|---|---|---|
| 2 | well list | `wells_163.csv` from notebook 1 |
| 3 | `mean_depth_to_water_m`, `elevation_m` | USGS daily depth to water (1980-2024 mean), USGS 3DEP elevation query |
| 4 | `mtpi`, `nlcd_code`, `landcover`, `landcover_simple`, `developed_intensity` | Google Earth Engine: CSP/ERGo mTPI (270 m), NLCD 2019 (30 m) |
| 5 | `clay`, `sand`, `silt` | SoilGrids, 0-5 cm, % |
| 6 | `swsc_mm`, `root_zone_storage_mm`, `bedrock_lith`, `PROVINCE` | Webb/Rosenzweig grids, USGS SGMC geology, physiographic provinces |
| 7 | `n_events`, `precip_gini`, `recharge_gini`, `total_precip_mm`, `total_recharge_mm`, `recharge_efficiency`, `kuma_a`, `kuma_b`, `kuma_shape` | notebook 1 event tables |
| 8 | saves `well_variables_163.csv` | same 27 columns and order as your master, plus `original_163` |

**Never touches `master_well_summary.csv`:** it is only read (optionally) at the very end to show how closely the rebuilt table matches the old one.

In [10]:
# ===== CELL 1: SETTINGS  =====
import os, io, json, time
import numpy as np, pandas as pd, geopandas as gpd, requests
from scipy.optimize import curve_fit

# ---- input files ----
WELLS_CSV   = "wells_163.csv"                                 
EVENT_DIR   = "dtw_event_tables_rpsy"                     
PHYSIO_PATH = r"C:\Users\romin\Downloads\physio_shp\physio.shp"                                                      # physiographic provinces
SGMC_PATH   = r"C:\Users\romin\Downloads\USGS_SGMC_Shapefiles\USGS_SGMC_Shapefiles\SGMC_Geology.shp"                # USGS geologic map (lithology)
WEBB_DATA   = r"C:\Users\romin\Downloads\WebbRosenzweig_548\WebbRosenzweig_548\data"                                 # Webb/Rosenzweig soil water storage grids
SWSC_ASC, ROOT_ASC = "wrprof.asc", "wrroot.asc"                                                                      # inside WEBB_DATA
GEE_PROJECT = "groundwater-project-507317"                                        

# ---- output ----
OUT_CSV = "well_variables_163.csv"
COMPARE_TO_MASTER = "master_well_summary.csv"               

# ---- caches ----
GWL_DIR, ELEV_DIR, MTPI_DIR, NLCD_DIR, SOIL_DIR = "gwl_raw", "elevation_cache", "mtpi_cache", "nlcd_cache", "soil_cache"
for d in (GWL_DIR, ELEV_DIR, MTPI_DIR, NLCD_DIR, SOIL_DIR): os.makedirs(d, exist_ok=True)

# ---- definitions ----
DEPTH_START, DEPTH_END = "1980-01-01", "2024-12-31"            # period of USGS daily depth averaged into mean depth to water
MIN_MAG = 5                                                    # storms under 5 mm excluded 

In [6]:
# ===== CELL 2: THE 163 WELLS =====
wells = pd.read_csv(WELLS_CSV, dtype={"well_id": str})[["well_id", "lat", "lon", "state"]].drop_duplicates("well_id").reset_index(drop=True)
V = wells.copy()                                             
print(len(V), "wells")

163 wells


In [7]:
# ===== CELL 3: MEAN DEPTH TO WATER (USGS daily record) AND ELEVATION (USGS 3DEP point query) =====
def get_usgs_gwl(site_id, start=DEPTH_START, end=DEPTH_END):
    """Daily depth to water (ft below land surface, USGS parameter 72019) -> GWL in m, negative below ground.
    Uses the daily mean, else median, else the average of daily max and min, else max or min alone. Cached in GWL_DIR."""
    path = f"{GWL_DIR}/{site_id}.csv"
    if os.path.exists(path):
        return pd.read_csv(path, parse_dates=["Date"]), "cached"
    for attempt in range(4):                                   # retry when the USGS server is busy
        r = requests.get("https://waterservices.usgs.gov/nwis/dv/", timeout=120,
                         params=dict(format="rdb", sites=site_id, startDT=start, endDT=end, parameterCd="72019", siteType="GW"))
        if r.status_code in (500, 502, 503, 504):
            time.sleep(5 * (attempt + 1)); continue
        break
    if r.status_code == 404:
        return None, "USGS returned no data (404)"
    r.raise_for_status()
    lines = [l for l in r.text.splitlines() if not l.startswith("#") and l.strip()]
    if len(lines) < 3:
        return None, "USGS returned an empty table"
    df = pd.read_csv(io.StringIO("\n".join(lines)), sep="\t", dtype=str).drop(index=0)
    val_cols = [c for c in df.columns if c not in ("agency_cd", "site_no", "datetime", "tz_cd") and not c.endswith("_cd")]
    def pick(stat):
        hit = [c for c in val_cols if c.endswith("_" + stat)]
        return pd.to_numeric(df[hit[0]], errors="coerce") if hit else None
    mean, med, mx, mn = pick("00003"), pick("00008"), pick("00001"), pick("00002")
    if mean is not None:   depth, used = mean, "mean"
    elif med is not None:  depth, used = med, "median"
    elif mx is not None and mn is not None: depth, used = (mx + mn) / 2, "avg of max and min"
    elif mx is not None:   depth, used = mx, "max only"
    elif mn is not None:   depth, used = mn, "min only"
    else: return None, f"no usable daily statistic (columns: {val_cols})"
    out = pd.DataFrame({"Date": pd.to_datetime(df["datetime"]), "GWL": -depth * 0.3048, "stat_used": used}).dropna()
    if len(out) == 0:
        return None, "all values non-numeric"
    out.to_csv(path, index=False)
    return out, "ok"

def get_elevation_cached(lat, lon):
    """Land-surface elevation (m) at the well from the USGS elevation point query service."""
    path = f"{ELEV_DIR}/{lat:.6f}_{lon:.6f}.json"
    if os.path.exists(path):
        return json.load(open(path))["elevation"]
    try:
        resp = requests.get("https://epqs.nationalmap.gov/v1/json", timeout=15,
                            params={"x": lon, "y": lat, "units": "Meters", "output": "json"})
        resp.raise_for_status()
        elevation = resp.json()["value"]
    except Exception as e:
        print(f"Elevation failed for ({lat}, {lon}): {e}")
        return None
    json.dump({"elevation": elevation}, open(path, "w"))
    return elevation

depth, elev = [], []
for i, r in V.iterrows():
    print(f"\r[{i + 1}/{len(V)}] {r['well_id']}", end="", flush=True)
    try:
        g, why = get_usgs_gwl(r["well_id"])
    except Exception as e:
        g, why = None, str(e)
    if g is None: print(f"\n  no depth for {r['well_id']}: {why}")
    depth.append(-g["GWL"].mean() if g is not None else np.nan)           # positive meters below the land surface
    elev.append(get_elevation_cached(r["lat"], r["lon"]))
V["mean_depth_to_water_m"] = depth
V["elevation_m"] = pd.to_numeric(pd.Series(elev), errors="coerce")
print("\n", V[["mean_depth_to_water_m", "elevation_m"]].describe().round(2).to_string())

[163/163] 453629068531801
        mean_depth_to_water_m  elevation_m
count                 163.00       163.00
mean                    5.52       174.19
std                     6.13       162.53
min                     0.46         2.86
25%                     2.01        42.22
50%                     3.56       128.61
75%                     6.40       276.87
max                    40.55       707.26


In [11]:
# ===== CELL 4: mTPI AND LAND USE (NLCD 2019) FROM GOOGLE EARTH ENGINE =====
import ee
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()                                      
    ee.Initialize(project=GEE_PROJECT)

def get_mtpi_cached(lat, lon):
    """Multiscale topographic position index (CSP/ERGo 270 m); negative = valley/low, positive = ridge/high."""
    path = f"{MTPI_DIR}/{lat:.6f}_{lon:.6f}.json"
    if os.path.exists(path):
        return json.load(open(path))["mtpi"]
    try:
        point = ee.Geometry.Point([lon, lat])
        value = ee.Image("CSP/ERGo/1_0/US/mTPI").select("elevation").reduceRegion(ee.Reducer.first(), point, scale=270).get("elevation")
        mtpi = value.getInfo()
    except Exception as e:
        print(f"mTPI failed for ({lat}, {lon}): {e}"); return None
    json.dump({"mtpi": mtpi}, open(path, "w"))
    return mtpi

def get_nlcd_landcover(lat, lon, year=2019):
    """NLCD land cover class code at the well (30 m)."""
    path = f"{NLCD_DIR}/{lat:.6f}_{lon:.6f}.json"
    if os.path.exists(path):
        return json.load(open(path))["code"]
    try:
        img = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD").filter(ee.Filter.eq("system:index", str(year))).first()
        point = ee.Geometry.Point([lon, lat])
        code = img.select("landcover").reduceRegion(ee.Reducer.first(), point, scale=30).get("landcover").getInfo()
    except Exception as e:
        print(f"NLCD failed for ({lat}, {lon}): {e}"); return None
    json.dump({"code": code}, open(path, "w"))
    return code

NLCD_CODES = {11: "Open Water", 12: "Perennial Ice/Snow", 21: "Developed, Open Space", 22: "Developed, Low Intensity",
              23: "Developed, Medium Intensity", 24: "Developed, High Intensity", 31: "Barren Land",
              41: "Deciduous Forest", 42: "Evergreen Forest", 43: "Mixed Forest", 51: "Dwarf Scrub", 52: "Shrub/Scrub",
              71: "Grassland/Herbaceous", 72: "Sedge/Herbaceous", 73: "Lichens", 74: "Moss",
              81: "Pasture/Hay", 82: "Cultivated Crops", 90: "Woody Wetlands", 95: "Emergent Herbaceous Wetlands"}
DEVELOPED_INTENSITY = {21: "Open Space", 22: "Low Intensity", 23: "Medium Intensity", 24: "High Intensity"}

def simplify_landcover(code):
    """Groups the detailed NLCD codes into broader categories."""
    if code is None or pd.isna(code): return None
    if code in [11, 12]: return "Water"
    if code in [21, 22, 23, 24]: return "Developed"
    if code == 31: return "Barren"
    if code in [41, 42, 43]: return "Forest"
    if code in [51, 52]: return "Shrub"
    if code in [71, 72, 73, 74]: return "Grassland"
    if code in [81, 82]: return "Agriculture"
    if code in [90, 95]: return "Wetland"
    return "Other"

mtpi, codes = [], []
for i, r in V.iterrows():
    print(f"\r[{i + 1}/{len(V)}] {r['well_id']}", end="", flush=True)
    mtpi.append(get_mtpi_cached(r["lat"], r["lon"]))
    codes.append(get_nlcd_landcover(r["lat"], r["lon"]))
V["mtpi"] = mtpi
V["nlcd_code"] = pd.Series(codes, dtype="float")
V["landcover"] = V["nlcd_code"].map(NLCD_CODES)
V["landcover_simple"] = V["nlcd_code"].map(simplify_landcover)
V["developed_intensity"] = V["nlcd_code"].map(DEVELOPED_INTENSITY)           # only filled for developed wells
print("\n", V["landcover_simple"].value_counts().to_string())

[163/163] 453629068531801
 landcover_simple
Developed      82
Forest         41
Agriculture    22
Wetland        14
Grassland       2
Barren          1
Shrub           1


In [12]:
# ===== CELL 5: SOIL TEXTURE (SoilGrids, 0-5 cm, % clay / sand / silt) =====
def get_soil_properties_cached(lat, lon, properties=("clay", "sand", "silt"), depth="0-5cm", retries=3, base_delay=2):
    path = f"{SOIL_DIR}/{lat:.6f}_{lon:.6f}.json"
    if os.path.exists(path):
        return json.load(open(path))
    params = {"lon": lon, "lat": lat, "property": list(properties), "depth": depth, "value": "mean"}
    for attempt in range(retries):
        resp = requests.get("https://rest.isric.org/soilgrids/v2.0/properties/query", params=params, timeout=30)
        if resp.status_code == 429:                            # rate limited: wait and retry
            time.sleep(base_delay * (attempt + 1)); continue
        if resp.status_code != 200:
            print(f"Soil failed for ({lat}, {lon}): status {resp.status_code}"); return None
        result = {}
        for layer in resp.json().get("properties", {}).get("layers", []):
            for d in layer.get("depths", []):
                if d["label"] == depth:
                    result[layer["name"]] = d["values"].get("mean")
        json.dump(result, open(path, "w"))
        return result
    print(f"Gave up on ({lat}, {lon}) after {retries} attempts."); return None

rows = []
for i, r in V.iterrows():
    print(f"\r[{i + 1}/{len(V)}] {r['well_id']}", end="", flush=True)
    new = not os.path.exists(f"{SOIL_DIR}/{r['lat']:.6f}_{r['lon']:.6f}.json")
    soil = get_soil_properties_cached(r["lat"], r["lon"])
    if soil is None or any(v is None for v in soil.values()) or len(soil) < 3:
        rows.append((np.nan, np.nan, np.nan))
    else:
        rows.append((soil["clay"] / 10, soil["sand"] / 10, soil["silt"] / 10))   # SoilGrids g/kg -> %
    if new: time.sleep(1.5)                                    # pace requests so the server does not rate-limit us
V[["clay", "sand", "silt"]] = pd.DataFrame(rows, index=V.index)
print("\n", V[["clay", "sand", "silt"]].describe().round(1).to_string())

[163/163] 453629068531801
         clay   sand   silt
count  131.0  131.0  131.0
mean    17.2   42.9   39.9
std      7.1   15.3    9.6
min      4.6   12.5   11.6
25%     10.6   31.0   35.2
50%     17.9   39.2   42.4
75%     23.0   53.5   46.5
max     33.2   83.0   59.2


In [17]:
# ===== CELL 6: SOIL WATER STORAGE (Webb/Rosenzweig grids), BEDROCK LITHOLOGY (USGS SGMC), PHYSIOGRAPHIC PROVINCE =====
def read_asc_grid(path):
    with open(path, "r") as f:
        header = {}
        expected = {"ncols", "nrows", "xllcorner", "yllcorner", "cellsize", "nodata_value"}
        while True:
            pos = f.tell()
            line = f.readline().split()
            if not line: continue
            key = line[0].lower()
            if key in expected: header[key] = float(line[1])
            else:
                f.seek(pos); break                           
        data = np.loadtxt(f)
    return data, header

def get_value_from_asc(lat, lon, data, header):
    """Value of the grid cell that contains (lat, lon)."""
    col = int((lon - header["xllcorner"]) / header["cellsize"])
    row = int((header["nrows"] - 1) - (lat - header["yllcorner"]) / header["cellsize"])
    if 0 <= row < data.shape[0] and 0 <= col < data.shape[1]:
        return data[row, col]
    return np.nan

swsc, swsc_h = read_asc_grid(os.path.join(WEBB_DATA, SWSC_ASC))
root, root_h = read_asc_grid(os.path.join(WEBB_DATA, ROOT_ASC))
V["swsc_mm"] = [get_value_from_asc(r["lat"], r["lon"], swsc, swsc_h) for _, r in V.iterrows()]
V["root_zone_storage_mm"] = [get_value_from_asc(r["lat"], r["lon"], root, root_h) for _, r in V.iterrows()]

pts = gpd.GeoDataFrame(V[["well_id"]].copy(), geometry=gpd.points_from_xy(V["lon"], V["lat"]), crs="EPSG:4326")

# bedrock lithology
sgmc = gpd.read_file(SGMC_PATH)
lith = gpd.sjoin(pts.to_crs(sgmc.crs), sgmc[["GENERALIZE", "geometry"]], how="left", predicate="within")     
lith = lith[["well_id", "GENERALIZE"]].rename(columns={"GENERALIZE": "bedrock_lith"}).drop_duplicates("well_id")   
V = V.drop(columns=["bedrock_lith"], errors="ignore").merge(lith, on="well_id", how="left")

# physiographic province
physio = gpd.read_file(PHYSIO_PATH)
physio = physio if physio.crs is not None else physio.set_crs(epsg=4326)
prov = gpd.sjoin(pts, physio.to_crs("EPSG:4326")[["PROVINCE", "geometry"]], how="left", predicate="within")
prov = prov[["well_id", "PROVINCE"]].drop_duplicates("well_id")
V = V.drop(columns=["PROVINCE"], errors="ignore").merge(prov, on="well_id", how="left")
print("missing bedrock lithology:", V["bedrock_lith"].isna().sum(), "| missing province:", V["PROVINCE"].isna().sum(), "| missing SWSC:", V["swsc_mm"].isna().sum())
print(V["PROVINCE"].value_counts().to_string())

missing bedrock lithology: 0 | missing province: 0 | missing SWSC: 0
PROVINCE
NEW ENGLAND             46
APPALACHIAN PLATEAUS    34
COASTAL PLAIN           31
VALLEY AND RIDGE        23
PIEDMONT                14
CENTRAL LOWLAND          9
ST. LAWRENCE VALLEY      4
BLUE RIDGE               2


In [19]:
print(ev.head())
print(ev.columns)

           well_id  n_events  precip_gini  recharge_gini  total_precip_mm  \
0  390211074505502       629     0.422143       0.645380         15466.14   
1  391145074520401       932     0.412594       0.597405         23730.16   
2  391621074435401       511     0.422218       0.526310         12972.78   
3  392232074234403       558     0.417484       0.532662         14962.17   
4  392731075092401      1567     0.411984       0.479545         40226.75   

   total_recharge_mm  recharge_efficiency    kuma_a    kuma_b  
0       51861.533906             3.353231  1.244154  1.020424  
1       54843.398509             2.311126  1.183529  1.004633  
2       68184.910503             5.255998  1.194271  1.186469  
3       41842.683059             2.796565  0.955194  0.885345  
4       99030.065603             2.461796  0.920795  0.983928  
Index(['well_id', 'n_events', 'precip_gini', 'recharge_gini',
       'total_precip_mm', 'total_recharge_mm', 'recharge_efficiency', 'kuma_a',
       'kum

In [20]:
# ===== CELL 7: VARIABLES FROM THE DTW EVENT TABLES: totals, recharge efficiency, Gini, Kumaraswamy a/b =====
def lorenz_curve(values):
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    cum_frac = np.cumsum(sorted_vals) / np.cumsum(sorted_vals)[-1]
    x = np.concatenate(([0], np.arange(1, n + 1) / n))
    y = np.concatenate(([0], cum_frac))
    trapezoid = getattr(np, "trapezoid", None) or np.trapz
    return x, y, 1 - 2 * trapezoid(y, x)                       

def kumaraswamy_cdf(x, a, b):
    return 1 - (1 - np.clip(x, 0, 1) ** a) ** b

def fit_kumaraswamy(x_data, y_data):
    try:
        popt, _ = curve_fit(kumaraswamy_cdf, x_data, y_data, p0=[1, 1], maxfev=5000)
        return popt
    except Exception:
        return None

def classify_shape(a, b):
    if a is None or b is None or pd.isna(a) or pd.isna(b): return None
    if a > 1 and b > 1:  return "S-shaped (mid-size storms dominate)"
    if a > 1 and b <= 1: return "Late-rise (few biggest storms dominate)"
    if a <= 1 and b > 1: return "Early-rise (small storms dominate)"
    return "U-shaped (both extremes matter, middle doesn't)"

def well_event_summary(table, min_mag=MIN_MAG, precip_col="precip_total", recharge_col="recharge_total"):
    df = table.dropna(subset=[precip_col, recharge_col]).copy()
    df = df[df[precip_col] >= min_mag]
    if len(df) < 2: return None
    _, _, precip_gini = lorenz_curve(df[precip_col].values)
    _, _, recharge_gini = lorenz_curve(df[recharge_col].values)
    total_precip, total_recharge = df[precip_col].sum(), df[recharge_col].sum() * 1000      # RpSy in m -> mm
    d = df.sort_values(precip_col, ascending=True).reset_index(drop=True)
    x = np.concatenate(([0], (d[precip_col].cumsum() / d[precip_col].sum()).values))
    y = np.concatenate(([0], (d[recharge_col].cumsum() / d[recharge_col].sum()).values))
    p = fit_kumaraswamy(x, y)
    a, b = (p[0], p[1]) if p is not None else (np.nan, np.nan)
    return dict(n_events=len(df), precip_gini=precip_gini, recharge_gini=recharge_gini, total_precip_mm=total_precip,
                total_recharge_mm=total_recharge, recharge_efficiency=total_recharge / total_precip if total_precip > 0 else np.nan,
                kuma_a=a, kuma_b=b)

rows = []
for wid in V["well_id"]:
    f = f"{EVENT_DIR}/{wid}_dtw_event_table.csv"
    s = well_event_summary(pd.read_csv(f)) if os.path.exists(f) else None
    rows.append({"well_id": wid, **(s or {})})
ev = pd.DataFrame(rows)
V = V.merge(ev, on="well_id", how="left")
V["kuma_shape"] = [
    classify_shape(a, b)
    for a, b in zip(V["kuma_a_y"], V["kuma_b_y"])
]
print("wells with event summaries:", V["n_events"].notna().sum(), "of", len(V))
print(V[["n_events", "precip_gini", "recharge_gini", "recharge_efficiency", "kuma_a", "kuma_b"]].describe().round(3).to_string())

wells with event summaries: 163 of 163
       n_events  precip_gini  recharge_gini  recharge_efficiency   kuma_a   kuma_b
count   163.000      163.000        163.000              163.000  163.000  163.000
mean    839.540        0.411          0.594                6.507    0.936    0.999
std     320.779        0.017          0.088                7.311    0.155    0.163
min     280.000        0.349          0.339                0.667    0.557    0.602
25%     609.000        0.400          0.540                2.422    0.809    0.885
50%     863.000        0.414          0.597                4.500    0.936    0.980
75%    1050.500        0.422          0.663                6.768    1.029    1.111
max    1668.000        0.451          0.786               48.249    1.342    1.581


In [21]:
# ===== CELL 8: SAVE THE VARIABLE TABLE (new file) AND COMPARE WITH MASTER  =====
ORDER = ["well_id", "mtpi", "elevation_m", "mean_depth_to_water_m", "clay", "sand", "silt", "state", "lat", "lon",
         "nlcd_code", "landcover", "landcover_simple", "kuma_shape", "swsc_mm", "root_zone_storage_mm", "bedrock_lith",
         "developed_intensity", "PROVINCE", "n_events", "precip_gini", "recharge_gini", "total_precip_mm",
         "total_recharge_mm", "recharge_efficiency", "kuma_a", "kuma_b"]
out = V[ORDER].copy()
out["original_163"] = True
out.to_csv(OUT_CSV, index=False)
print(f"saved {OUT_CSV}: {out.shape[0]} wells x {out.shape[1]} columns")
miss = out.isna().sum(); miss = miss[miss > 0]
print("missing values per column:", "none" if miss.empty else "\n" + miss.to_string())

if COMPARE_TO_MASTER and os.path.exists(COMPARE_TO_MASTER):
    old = pd.read_csv(COMPARE_TO_MASTER, dtype={"well_id": str}).drop_duplicates("well_id").set_index("well_id")
    new = out.set_index("well_id")
    both = [w for w in new.index if w in old.index]
    print(f"\nComparison with {COMPARE_TO_MASTER} ({len(both)} wells in both):")
    for c in ORDER[1:]:
        if c not in old.columns: continue
        if pd.api.types.is_numeric_dtype(new[c]) and pd.api.types.is_numeric_dtype(old[c]):
            d = (new.loc[both, c].astype(float).values - old.loc[both, c].astype(float).values)
            d = np.abs(d)
            print(f"  {c:24s} max abs difference = {np.nanmax(d):.4g}   same for {np.mean(d < 1e-6) * 100:.0f}% of wells")
        else:
            same = new.loc[both, c].astype(str).values == old.loc[both, c].astype(str).values
            print(f"  {c:24s} identical text for {same.mean() * 100:.0f}% of wells")

saved well_variables_163.csv: 163 wells x 28 columns
missing values per column: 
clay                   32
sand                   32
silt                   32
developed_intensity    81

Comparison with master_well_summary.csv (163 wells in both):
  mtpi                     max abs difference = 0   same for 100% of wells
  elevation_m              max abs difference = 0   same for 100% of wells
  mean_depth_to_water_m    max abs difference = 1.547   same for 2% of wells
  clay                     max abs difference = 0   same for 80% of wells
  sand                     max abs difference = 0   same for 80% of wells
  silt                     max abs difference = 0   same for 80% of wells
  state                    identical text for 100% of wells
  lat                      max abs difference = 0   same for 100% of wells
  lon                      max abs difference = 0   same for 100% of wells
  nlcd_code                max abs difference = 0   same for 100% of wells
  landcover        